# 08 — Customer Segmentation

**Objective:** Segment customers using K-Means and Hierarchical clustering based on FVS, flights frequency, redemption ratio, and churn probability. Profile and assign business-friendly names to each segment.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

from src.config import *
from src.utils import logger, timer, save_csv, save_model

## 1. Prepare Segmentation Data

Load data, scale features, and combine model-predicted churn probabilities for multi-dimensional clustering.

In [2]:
SEGMENTATION_FEATURES = [
    "Flights_12M",
    "Redemption_Ratio",
    "FVS",
]

def prepare_segmentation_data(
    features_path: Optional[Path] = None,
) -> tuple:
    if features_path is None:
        features_path = FEATURES_DIR / "customer_features_with_fvs.csv"

    df = pd.read_csv(features_path)

    try:
        churn_probs = pd.read_csv(FEATURES_DIR / "churn_predictions.csv")
        df = df.merge(churn_probs[[PK, "Churn_Probability"]], on=PK, how="left")
        df["Churn_Probability"] = df["Churn_Probability"].fillna(df["Churn"])
    except FileNotFoundError:
        df["Churn_Probability"] = df["Churn"].astype(float)
        logger.warning("Using actual churn labels as proxy for churn probability")

    seg_features = ["Churn_Probability"] + SEGMENTATION_FEATURES
    X_seg = df[seg_features].fillna(0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_seg)

    return df, X_seg, X_scaled, scaler, seg_features

df, X_seg, X_scaled, scaler, seg_features = prepare_segmentation_data()

## 2. K-Means and Hierarchical Clustering

Apply clustering models to partition the customers and assess silhouette scores to identify the optimal cluster count.

In [4]:
def run_kmeans(X_scaled: np.ndarray, k_range: range = range(3, 9)) -> dict:
    results = {}
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10, max_iter=300)
        labels = kmeans.fit_predict(X_scaled)
        sil_score = silhouette_score(X_scaled, labels)
        inertia = kmeans.inertia_

        results[k] = {
            "model": kmeans,
            "labels": labels,
            "silhouette": sil_score,
            "inertia": inertia,
        }
        logger.info(f"K={k}: Silhouette={sil_score:.3f}, Inertia={inertia:.0f}")
    return results

def run_hierarchical(X_scaled: np.ndarray, n_clusters: int = 5) -> dict:
    model = AgglomerativeClustering(
        n_clusters=n_clusters, linkage="ward"
    )
    labels = model.fit_predict(X_scaled)
    sil_score = silhouette_score(X_scaled, labels)
    logger.info(f"Hierarchical (K={n_clusters}): Silhouette={sil_score:.3f}")
    return {
        "model": model,
        "labels": labels,
        "silhouette": sil_score,
    }

def determine_optimal_k(kmeans_results: dict) -> int:
    best_k = max(kmeans_results.keys(), key=lambda k: kmeans_results[k]["silhouette"])
    logger.info(f"Optimal K by silhouette: {best_k}")
    return best_k

kmeans_results = run_kmeans(X_scaled)
optimal_k = determine_optimal_k(kmeans_results)

2026-06-12 11:26:12 | airline_loyalty | INFO | K=3: Silhouette=0.316, Inertia=32489
2026-06-12 11:26:16 | airline_loyalty | INFO | K=4: Silhouette=0.335, Inertia=26772
2026-06-12 11:26:19 | airline_loyalty | INFO | K=5: Silhouette=0.306, Inertia=22979
2026-06-12 11:26:23 | airline_loyalty | INFO | K=6: Silhouette=0.309, Inertia=19599
2026-06-12 11:26:26 | airline_loyalty | INFO | K=7: Silhouette=0.280, Inertia=17978
2026-06-12 11:26:29 | airline_loyalty | INFO | K=8: Silhouette=0.285, Inertia=16455
2026-06-12 11:26:29 | airline_loyalty | INFO | Optimal K by silhouette: 4


## 3. Cluster Profiling and Naming

Map clusters to qualitative business personas based on their average FVS, flight activity, and churn profiles.

In [5]:
def profile_clusters(
    df: pd.DataFrame, labels: np.ndarray, seg_features: list
) -> pd.DataFrame:
    df_clustered = df.copy()
    df_clustered["Cluster"] = labels
    profile_cols = seg_features + ["CLV", "Tenure_Months", "Flights_Lifetime"]
    available_cols = [c for c in profile_cols if c in df_clustered.columns]

    profiles = df_clustered.groupby("Cluster")[available_cols].mean().round(2)
    profiles["Count"] = df_clustered.groupby("Cluster").size()
    profiles["Pct"] = (profiles["Count"] / len(df_clustered) * 100).round(1)
    return profiles

def name_clusters(profiles: pd.DataFrame) -> dict:
    names = {}
    for cluster_id in profiles.index:
        row = profiles.loc[cluster_id]
        churn_risk = row.get("Churn_Probability", 0.5)
        fvs = row.get("FVS", 50)
        flights = row.get("Flights_12M", 0)

        if fvs > 60 and churn_risk < 0.3:
            names[cluster_id] = "Champions"
        elif fvs > 50 and churn_risk > 0.5:
            names[cluster_id] = "VIP At Risk"
        elif fvs > 30 and churn_risk < 0.4 and flights > 5:
            names[cluster_id] = "Loyal Travelers"
        elif fvs > 20 and churn_risk < 0.5:
            names[cluster_id] = "Growth Potential"
        else:
            names[cluster_id] = "Dormant Members"

    # Handle duplicates by applying unique fallback values
    seen = {}
    for k, v in names.items():
        if v in seen.values():
            alternatives = ["Champions", "VIP At Risk", "Loyal Travelers",
                          "Growth Potential", "Dormant Members"]
            used = set(seen.values())
            for alt in alternatives:
                if alt not in used:
                    names[k] = alt
                    break
        seen[k] = names[k]

    logger.info(f"Cluster names: {names}")
    return names

## 4. Save Segmented Profiles

Run final segmentation pipeline with 5 clusters (matching executive specifications), assign profiles, and export the dataset.

In [6]:
k = 5
if k in kmeans_results:
    final_labels = kmeans_results[k]["labels"]
    final_model = kmeans_results[k]["model"]
else:
    final_model = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    final_labels = final_model.fit_predict(X_scaled)

hier_result = run_hierarchical(X_scaled, k)
profiles = profile_clusters(df, final_labels, seg_features)
cluster_names = name_clusters(profiles)

df["Cluster"] = final_labels
df["Segment"] = df["Cluster"].map(cluster_names)

save_csv(df, FEATURES_DIR / "customer_segmented.csv")
save_model(final_model, MODELS_DIR / "segmentation_model.joblib")
save_model(scaler, MODELS_DIR / "segmentation_scaler.joblib")

print(profiles)

2026-06-12 11:27:00 | airline_loyalty | INFO | Hierarchical (K=5): Silhouette=0.270
2026-06-12 11:27:00 | airline_loyalty | INFO | Cluster names: {0: 'Loyal Travelers', 1: 'Champions', 2: 'Dormant Members', 3: 'VIP At Risk', 4: 'Growth Potential'}
2026-06-12 11:27:00 | airline_loyalty | INFO | Saved: customer_segmented.csv (16,737 rows × 55 cols)
2026-06-12 11:27:00 | airline_loyalty | INFO | Saved model: segmentation_model.joblib
2026-06-12 11:27:00 | airline_loyalty | INFO | Saved model: segmentation_scaler.joblib


         Churn_Probability  Flights_12M  Redemption_Ratio    FVS       CLV  \
Cluster                                                                      
0                     0.10        12.32              0.06  33.74   7061.29   
1                     0.04        17.12              0.01  33.70   6168.09   
2                     0.96         2.79              0.00  22.90   7825.17   
3                     0.15         3.47              0.00  18.96   7159.00   
4                     0.04        21.94              0.02  56.15  10796.85   

         Tenure_Months  Flights_Lifetime  Count   Pct  
Cluster                                                
0                34.82             17.74   1199   7.2  
1                39.26             24.80   6156  36.8  
2                18.25              4.64   2617  15.6  
3                10.97              4.63   2146  12.8  
4                51.66             30.90   4619  27.6  
